In [1]:
import os
import subprocess
from pathlib import Path
from datetime import datetime, timedelta

# === INPUT USER ===
input_folder = input("Masukkan path folder video (misal: E:/dataset/project/tanggal/2.360_mp4): ").strip('"')
fps = input("Masukkan FPS yang diinginkan (misal: 6): ").strip()

# Parent = E:/dataset/project/tanggal/
parent_dir = str(Path(input_folder).parent)
output_root = os.path.join(parent_dir, "3.frame_360", fps)
os.makedirs(output_root, exist_ok=True)

# === FUNGSI BANTUAN ===
def get_video_duration(video_path):
    """Mengambil durasi video dalam detik menggunakan ffprobe"""
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        video_path
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return float(result.stdout.strip())
    except subprocess.CalledProcessError:
        return None

# === PROSES LOOP VIDEO ===
video_files = [f for f in os.listdir(input_folder) if f.lower().endswith(".mp4")]
video_files.sort()

summary_data = []

print(f"\nDitemukan {len(video_files)} file video di folder {input_folder}\n")

for idx, video_name in enumerate(video_files, start=1):
    video_path = os.path.join(input_folder, video_name)
    duration = get_video_duration(video_path)
    if duration is None:
        print(f"[{idx}] Gagal membaca durasi: {video_name}")
        continue

    start_time = datetime.now()
    print(f"[{idx}] Memproses: {video_name}")
    print(f"     Durasi : {timedelta(seconds=round(duration))}")
    print(f"     Mulai  : {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

    # === OUTPUT PER VIDEO ===
    video_stem = Path(video_name).stem
    video_output_dir = os.path.join(output_root, video_stem)
    os.makedirs(video_output_dir, exist_ok=True)

    # output_pattern = os.path.join(video_output_dir, "image_%04d.jpg")
    output_pattern = os.path.join(video_output_dir, f"{video_stem}_image_%04d.jpg")

    # Jalankan ffmpeg
    cmd = [
        "ffmpeg", "-i", video_path,
        "-vf", f"fps={fps}",
        "-qscale:v", "1",
        output_pattern,
        "-hide_banner", "-loglevel", "error"
    ]
    subprocess.run(cmd)

    end_time = datetime.now()
    total_frames = len([f for f in os.listdir(video_output_dir) if f.endswith(".jpg")])

    print(f"     Selesai: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"     Total Frame: {total_frames}")
    print("-"*60 + "\n")

    summary_data.append({
        "no": idx,
        "nama_video": video_name,
        "durasi_detik": round(duration, 2),
        "total_frame": total_frames
    })

print("\n=== Semua Proses Selesai ===\n")

# === SIMPAN SUMMARY ===
summary_filename = f"summary_txt_fps{fps}.txt"
summary_path = os.path.join(output_root, summary_filename)

with open(summary_path, "w") as f:
    f.write(f"Summary Ekstraksi Frame (FPS = {fps})\n")
    f.write("="*50 + "\n\n")
    for s in summary_data:
        f.write(f"Video ke-{s['no']}\n")
        f.write(f"Nama        : {s['nama_video']}\n")
        f.write(f"Durasi      : {s['durasi_detik']} detik\n")
        f.write(f"Total Frame : {s['total_frame']}\n")
        f.write("-"*50 + "\n")

print(f"Summary disimpan di: {summary_path}")



Ditemukan 2 file video di folder D:\metadata\360\GAGAL\2.360_mp4

[1] Memproses: GS014244.mp4
     Durasi : 0:01:00
     Mulai  : 2026-01-29 14:52:15
     Selesai: 2026-01-29 14:53:29
     Total Frame: 180
------------------------------------------------------------

[2] Memproses: GS014287.mp4
     Durasi : 0:01:04
     Mulai  : 2026-01-29 14:53:29
     Selesai: 2026-01-29 14:54:42
     Total Frame: 192
------------------------------------------------------------


=== Semua Proses Selesai ===

Summary disimpan di: D:\metadata\360\GAGAL\3.frame_360\3\summary_txt_fps3.txt
